<a href="https://colab.research.google.com/github/naamasarshalom-art/segmentation_cellpose/blob/main/3_RADIO_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RADIO Embeddings — Feature Extraction for Model 1

This notebook extracts image embeddings from the **C-RADIOv4-H** foundation model (NVIDIA)
for all images in the labeled dataset.

The resulting embeddings are used to train **Model 1**: a binary classifier that separates
classifiable nuclei from trash/artifacts.

**Pipeline:**
1. Load the C-RADIOv4-H model (requires GPU)
2. Collect images from all four class folders
3. Extract a 2560-dimensional summary embedding per image
4. Save embeddings, labels, and paths to a `.npz` file
5. Visualize the embedding space with PCA (2D)

**Classes:**
- `0` — good
- `1` — invaginated
- `2` — Unclassifiable
- `3` — trash

## 1. Setup — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

import torch
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor
from tqdm.auto import tqdm
from sklearn.decomposition import PCA

## 3. Configuration

Set the dataset path here. All other cells read from these variables.

In [ ]:
# ── Dataset paths ──────────────────────────────────────────────────────────
DATASET_DIR = "/content/drive/MyDrive/model_nuc/dataset v.0.3"  # ← change if needed

CLASS_DIRS = {
    0: f"{DATASET_DIR}/good",
    1: f"{DATASET_DIR}/invaginated",
    2: f"{DATASET_DIR}/Unclassifiable",
    3: f"{DATASET_DIR}/trash",
}

CLASS_NAMES = {0: "good", 1: "invaginated", 2: "Unclassifiable", 3: "trash"}

# ── Output path ────────────────────────────────────────────────────────────
OUTPUT_NPZ = "/content/drive/MyDrive/model_nuc/embeddings_radio.npz"

# ── RADIO model version ────────────────────────────────────────────────────
MODEL_VERSION = "c-radio_v4-h"

# ── Supported image extensions ─────────────────────────────────────────────
EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.tif", "*.tiff", "*.bmp", "*.webp")

## 4. Load RADIO Model

Requires GPU. If this cell fails, go to Runtime → Change runtime type → GPU.

In [ ]:
device = "cuda"

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → GPU.")

# Load C-RADIOv4-H from the NVlabs hub (~1.57 GB, downloaded once and cached)
radio_model = torch.hub.load(
    'NVlabs/RADIO',
    'radio_model',
    version=MODEL_VERSION,
    skip_validation=True,
    force_reload=False
)
radio_model.to(device).eval()
print(f"✓ RADIO model loaded: {MODEL_VERSION}")

## 5. Collect Images from Dataset

In [ ]:
def collect_images(dir_path):
    """Return a list of all image file paths in dir_path."""
    paths = []
    for ext in EXTENSIONS:
        paths.extend(glob.glob(os.path.join(dir_path, ext)))
    return sorted(paths)


all_paths  = []
all_labels = []

for label, dir_path in CLASS_DIRS.items():
    imgs = collect_images(dir_path)
    all_paths.extend(imgs)
    all_labels.extend([label] * len(imgs))
    print(f"  {CLASS_NAMES[label]:>15}: {len(imgs)} images")

all_labels = np.array(all_labels)
print(f"\n  Total: {len(all_paths)} images")

## 6. Extract RADIO Embeddings

Each image is resized to the nearest resolution supported by the model,
then passed through RADIO to produce a 2560-dimensional summary vector.

In [ ]:
@torch.no_grad()
def get_radio_embeddings(image_paths, batch_size=8):
    """
    Extract RADIO summary embeddings for a list of image paths.

    All images in a batch are resized to the same resolution — determined
    by the first image in the batch via get_nearest_supported_resolution().

    Returns:
        numpy array of shape (N, 2560)
    """
    all_embs = []

    for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting embeddings"):
        batch_paths = image_paths[i : i + batch_size]

        # Load images as PIL and find the target resolution from the first image
        pil_imgs  = [Image.open(p).convert("RGB") for p in batch_paths]
        w, h      = pil_imgs[0].size
        target_hw = radio_model.get_nearest_supported_resolution(h, w)

        # Resize all images in batch to the same resolution, convert to tensor
        tensors = [
            pil_to_tensor(img.resize((target_hw[1], target_hw[0]), Image.BILINEAR)).float() / 255.0
            for img in pil_imgs
        ]

        x = torch.stack(tensors).to(device)

        # Forward pass — summary is the global CLS-like token (shape: B x 2560)
        summary, _ = radio_model(x)
        all_embs.append(summary.cpu().numpy())

    return np.vstack(all_embs)


X = get_radio_embeddings(all_paths, batch_size=8)
print(f"\n✓ Embeddings shape: {X.shape}")

## 7. Save Embeddings to Disk

In [ ]:
# Save embeddings, integer labels, and source paths together in one .npz file.
# This file is the input for the Model 1 training notebook.
np.savez(
    OUTPUT_NPZ,
    X      = X,
    labels = all_labels,
    paths  = np.array(all_paths)
)
print(f"✓ Saved to: {OUTPUT_NPZ}")

## 8. Visualize Embedding Space with PCA

Project the 2560-dimensional embeddings to 2D using PCA.
A clean separation between trash and classifiable classes suggests
the embeddings carry enough signal for Model 1.

In [ ]:
pca = PCA(n_components=2, random_state=0)
X2  = pca.fit_transform(X)
print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}  PC2={pca.explained_variance_ratio_[1]:.1%}")

COLORS = {0: "steelblue", 1: "tomato", 2: "goldenrod", 3: "gray"}

plt.figure(figsize=(8, 6))
for label, name in CLASS_NAMES.items():
    mask = all_labels == label
    plt.scatter(X2[mask, 0], X2[mask, 1],
                label=name, alpha=0.7, s=20, color=COLORS[label])

plt.title("PCA of RADIO Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()